# Demo 0: Example and usage

In order to make things simple the following rules have been followed
during development:

-   `deel-lip` follows the `keras` package structure.
-   All elements (layers, activations, initializers, ...) are compatible
    with standard the `keras` elements.
-   When a k-Lipschitz layer overrides a standard keras layer, it uses
    the same interface and the same parameters. The only difference is a
    new parameter to control the Lipschitz constant of a layer.

## Which layers are safe to use?

The following table indicates which layers are safe to use in a Lipshitz
network, and which are not.

| layer                                                                                         | 1-lip? | deel-lip equivalent                                                                                         | comments                                                                          |
|-----------------------------------------------------------------------------------------------|--------|-------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------|
| `Dense`                                                                                       | no     | `SpectralDense`<br>`FrobeniusDense`                       | `SpectralDense` and `FrobeniusDense` are similar when there is a single output. |
| `Conv2D`                                                                                      | no     | `SpectralConv2D`<br>`FrobeniusConv2D`                     | `SpectralConv2D` also implements Björck normalization.                           |
| `MaxPooling`<br>`GlobalMaxPooling`             | yes    | n/a                                                                                                         |                                                                                   |
| `AveragePooling2D`<br>`GlobalAveragePooling2D` | no     | `ScaledAveragePooling2D`<br>`ScaledGlobalAveragePooling2D` | The lipschitz constant is bounded by `sqrt(pool_h * pool_h)`.                     |
| `Flatten`                                                                                     | yes    | n/a                                                                                                         |                                                                                   |
| `Dropout`                                                                                     | no     | None                                                                                                        | The lipschitz constant is bounded by the dropout factor.                          |
| `BatchNormalization`                                                                          | no     | None                                                                                                        | We suspect that layer normalization already limits internal covariate shift.      |

## Design tips

Designing lipschitz networks requires a careful design in order to avoid
vanishing/exploding gradient problems.

Choosing pooling layers:

| layer                                                                            | advantages                                                                   | disadvantages                                                                      |
|----------------------------------------------------------------------------------|------------------------------------------------------------------------------|------------------------------------------------------------------------------------|
| `ScaledAveragePooling2D` and `MaxPooling2D`                                      | very similar to original implementation (just add a scaling factor for avg). | not norm preserving nor gradient norm preserving.                                  |
| `InvertibleDownSampling`                                                         | norm preserving and gradient norm preserving.                                | increases the number of channels (and the number of parameters of the next layer). |
| `ScaledL2NormPooling2D` (_sqrt(avgpool(x\*\*2))_) | norm preserving.                                                             | lower numerical stability of the gradient when inputs are close to zero.           |

Choosing activations:

| layer                                                                  | advantages                                                                                   | disadvantages                                                                                  |
|------------------------------------------------------------------------|----------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------|
| `ReLU`                                                                 |                                                                                              | create a strong vanishing gradient effect. If you manage to learn with it, please call 911.    |
| `MaxMin` (_stack(\[ReLU(x), ReLU(-x)\])_) | have similar properties to ReLU, but is norm and gradient norm preserving                    | double the number of outputs                                                                   |
| `GroupSort`                                                            | Input and GradientNorm preserving. Also limit the need of biases (as it is shift invariant). | more computationally expensive, (when its parameter _n_ is large) |

Please note that when learning with the `HKR_loss` and `HKR_multiclass_loss`, no
activation is required on the last layer.


### How to use it ?
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deel-ai/deel-lip/blob/master/docs/notebooks/demo0.ipynb)

Here is an example of 1-lipschitz network trained on MNIST:

In [21]:
from deel.lip.layers import (
    SpectralDense,
    SpectralConv2D,
    ScaledL2NormPooling2D,
    FrobeniusDense,
)
from deel.lip.model import Sequential
from deel.lip.activations import GroupSort
from deel.lip.losses import MulticlassHKR, MulticlassKR
from keras.layers import Input, Flatten
from keras.optimizers import Adam
from keras.datasets import mnist
from keras.utils import to_categorical
import keras
import numpy as np

In [ ]:
# # load data
# (x_train, y_train), (x_test, y_test) = mnist.load_data()
# # standardize and reshape the data
# x_train = np.expand_dims(x_train, -1)
# mean = x_train.mean()
# std = x_train.std()
# x_train = (x_train - mean) / std
# x_test = np.expand_dims(x_test, -1)
# x_test = (x_test - mean) / std
# # one hot encode the labels
# y_train = to_categorical(y_train)
# y_test = to_categorical(y_test)
 

In [2]:
# load data
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# standardize and reshape the data
x_train = np.expand_dims(x_train, -1) / 255
x_test = np.expand_dims(x_test, -1) / 255
# one hot encode the labels
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [3]:
x_train = np.transpose(x_train,(0,3,1,2))
x_test = np.transpose(x_test,(0,3,1,2))

In [4]:
x_train.shape

(60000, 1, 28, 28)

In [5]:
np.min(x_train)

0.0

In [13]:
# Sequential (resp Model) from deel.model has the same properties as any lipschitz model.
# It act only as a container, with features specific to lipschitz
# functions (condensation, vanilla_exportation...) but The layers are fully compatible
# with the tf.keras.model.Sequential/Model
model = Sequential(
    [
        Input(shape=x_train.shape[1:]),
        # Lipschitz layers preserve the API of their superclass ( here Conv2D )
        # an optional param is available: k_coef_lip which control the lipschitz
        # constant of the layer
        SpectralConv2D(
            filters=16,
            kernel_size=(3, 3),
            activation=GroupSort(2),
            use_bias=True,
            kernel_initializer="orthogonal",
        ),
        # usual pooling layer are implemented (avg, max...), but new layers are also available
        ScaledL2NormPooling2D(pool_size=(2, 2), data_format="channels_first"),
        SpectralConv2D(
            filters=16,
            kernel_size=(3, 3),
            activation=GroupSort(2),
            use_bias=True,
            kernel_initializer="orthogonal",
        ),
        ScaledL2NormPooling2D(pool_size=(2, 2), data_format="channels_first"),
        # our layers are fully interoperable with existing keras layers
        Flatten(data_format="channels_last"),
        # SpectralDense(
        #     32,
        #     activation=GroupSort(2),
        #     use_bias=True,
        #     kernel_initializer="orthogonal",
        # ),
        # SpectralDense(
        #     10, activation=None, use_bias=False, kernel_initializer="orthogonal"
        # ),
    ],
    # similary model has a parameter to set the lipschitz constant
    # to set automatically the constant of each layer
    k_coef_lip=1.0,
    name="hkr_model",
)
model.summary()

Model: "hkr_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ spectral_conv2d_6               │ (None, 16, 28, 28)     │           321 │
│ (SpectralConv2D)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ scaled_l2_norm_pooling2d_6      │ (None, 16, 14, 14)     │             0 │
│ (ScaledL2NormPooling2D)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spectral_conv2d_7               │ (None, 16, 14, 14)     │         4,641 │
│ (SpectralConv2D)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ scaled_l2_norm_pooling2d_7      │ (None, 16, 7, 7)       │             0 │
│ (ScaledL2NormPooling2D)         │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 784)            │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,962 (19.38 KB)

 Trainable params: 2,480 (9.69 KB)

 Non-trainable params: 2,482 (9.70 KB)

In [30]:
x = keras.random.normal((16,7,7))[None]

In [31]:
Flatten(data_format="channels_first")(x)

<tf.Tensor: shape=(1, 784), dtype=float32, numpy=
array([[ 1.28813899e+00, -9.97370064e-01,  4.25578421e-03,
        -5.64867496e-01, -9.81105864e-01,  6.64906502e-01,
        -1.56951106e+00,  2.39926791e+00,  1.48891866e+00,
         4.72740978e-02, -3.15707636e+00, -1.71449625e+00,
         1.74879837e+00,  2.32356548e-01,  7.86637723e-01,
         3.13740999e-01,  4.08251554e-01,  6.52428210e-01,
        -2.15210125e-01,  7.79724240e-01, -5.74635744e-01,
        -1.60046518e+00, -2.73461604e+00, -6.00960791e-01,
        -1.43342525e-01, -5.11284828e-01, -1.01470363e+00,
        -1.20736456e+00, -1.97175097e+00, -1.51815450e+00,
         7.67515421e-01,  2.01104736e+00, -1.89649478e-01,
        -3.30569029e-01, -6.53753996e-01, -2.76339650e-01,
        -2.12800860e+00,  3.70083630e-01,  7.26212323e-01,
         2.09974647e+00,  8.43753815e-01, -4.98599023e-01,
         3.51676226e-01, -8.90019417e-01,  3.82755160e-01,
        -3.30787361e-01,  5.48129082e-01, -1.72320664e-01,
      

In [32]:
Flatten(data_format="channels_last")(x)

<tf.Tensor: shape=(1, 784), dtype=float32, numpy=
array([[ 1.28813899e+00,  4.08251554e-01, -1.89649478e-01,
         1.16380858e+00, -1.84982419e+00,  5.52642286e-01,
        -8.99601102e-01, -6.72446370e-01, -7.68095791e-01,
        -1.81756407e-01,  5.08753121e-01, -6.72164142e-01,
         7.02419579e-01,  8.26343179e-01, -2.87392233e-02,
        -4.67153311e-01,  2.42894381e-01,  4.67088908e-01,
        -1.26013267e+00, -1.41597772e+00,  1.20028186e+00,
         8.46293092e-01, -4.44624096e-01,  4.16112810e-01,
         1.38455129e+00,  3.38928103e-01, -2.09511489e-01,
         2.45806500e-01, -5.64989865e-01,  3.28011721e-01,
        -1.08922578e-01, -4.71754998e-01,  8.75010312e-01,
         1.08323562e+00,  6.59535170e-01, -4.87078726e-02,
         3.44357848e-01, -3.22185421e+00,  5.72564960e-01,
         3.48603874e-01,  1.17553389e+00,  1.27394867e+00,
        -2.13153481e-01, -1.64469317e-01, -2.72128582e-01,
        -4.71404701e-01,  1.42996192e+00,  1.43724453e+00,
      

In [33]:
Flatten()(x)

<tf.Tensor: shape=(1, 784), dtype=float32, numpy=
array([[ 1.28813899e+00, -9.97370064e-01,  4.25578421e-03,
        -5.64867496e-01, -9.81105864e-01,  6.64906502e-01,
        -1.56951106e+00,  2.39926791e+00,  1.48891866e+00,
         4.72740978e-02, -3.15707636e+00, -1.71449625e+00,
         1.74879837e+00,  2.32356548e-01,  7.86637723e-01,
         3.13740999e-01,  4.08251554e-01,  6.52428210e-01,
        -2.15210125e-01,  7.79724240e-01, -5.74635744e-01,
        -1.60046518e+00, -2.73461604e+00, -6.00960791e-01,
        -1.43342525e-01, -5.11284828e-01, -1.01470363e+00,
        -1.20736456e+00, -1.97175097e+00, -1.51815450e+00,
         7.67515421e-01,  2.01104736e+00, -1.89649478e-01,
        -3.30569029e-01, -6.53753996e-01, -2.76339650e-01,
        -2.12800860e+00,  3.70083630e-01,  7.26212323e-01,
         2.09974647e+00,  8.43753815e-01, -4.98599023e-01,
         3.51676226e-01, -8.90019417e-01,  3.82755160e-01,
        -3.30787361e-01,  5.48129082e-01, -1.72320664e-01,
      

In [8]:
# HKR (Hinge-Krantorovich-Rubinstein) optimize robustness along with accuracy
model.compile(
    # decreasing alpha and increasing min_margin improve robustness (at the cost of accuracy)
    # note also in the case of lipschitz networks, more robustness require more parameters.
    loss=MulticlassHKR(alpha=50, min_margin=0.05),
    optimizer=Adam(1e-3),
    metrics=["accuracy", MulticlassKR()],
)

In [9]:
# fit the model
model.fit(
    x_train,
    y_train,
    batch_size=2048,
    epochs=100,
    validation_data=(x_test, y_test),
    shuffle=True,
)

Epoch 1/100


I0000 00:00:1743760434.730049   22345 service.cc:148] XLA service 0x55ebbcdf1870 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1743760434.730106   22345 service.cc:156]   StreamExecutor device (0): NVIDIA A10G, Compute Capability 8.6
2025-04-04 11:53:54.807238: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1743760435.098507   22345 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-04-04 11:53:55.290336: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.4 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-04-04 11:53:57.377886: I 

17/30 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - MulticlassKR: 0.0505 - accuracy: 0.3450 - loss: 2.3751

I0000 00:00:1743760444.348308   22345 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 173ms/step - MulticlassKR: 0.0759 - accuracy: 0.4678 - loss: 1.7830 - val_MulticlassKR: 0.2074 - val_accuracy: 0.8879 - val_loss: 0.1568
Epoch 2/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - MulticlassKR: 0.2320 - accuracy: 0.8972 - loss: 0.1052 - val_MulticlassKR: 0.3192 - val_accuracy: 0.9296 - val_loss: -0.0712
Epoch 3/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - MulticlassKR: 0.3485 - accuracy: 0.9305 - loss: -0.0981 - val_MulticlassKR: 0.4753 - val_accuracy: 0.9446 - val_loss: -0.2501
Epoch 4/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - MulticlassKR: 0.5230 - accuracy: 0.9433 - loss: -0.2786 - val_MulticlassKR: 0.7108 - val_accuracy: 0.9473 - val_loss: -0.4603
Epoch 5/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - MulticlassKR: 0.7692 - accuracy: 0.9466 - loss: -0.5050 - val_MulticlassKR: 0.9901 - val_accuracy: 0.9532 - val_loss: -0.7068
Epoch 6/100
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - MulticlassKR: 1.0475 - accuracy: 0.9499 - loss: -0.7331 - v

In [13]:
vanilla_model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - MulticlassKR: 2.4034 - accuracy: 0.9600 - loss: -1.9218


[-2.073857307434082, 0.9672999978065491, 2.5064444541931152]

In [10]:

# once training is finished you can convert
# SpectralDense layers into Dense layers and SpectralConv2D into Conv2D
# which optimize performance for inference
vanilla_model = model.vanilla_export()

In [11]:
model.save('/home/aws_install/robustess_project/lip_models/demo0_MNIST_channelfirst_False_disj_Neurons.keras')
vanilla_model.save("/home/aws_install/robustess_project/lip_models/demo0_vanilla_MNIST_channelfirst_False_disj_Neurons.keras")